<a href="https://colab.research.google.com/github/gp28896/Hybrid-Encryption-Python/blob/main/Encryption_Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install cryptography pyjwt pandas matplotlib

1. Introduction
2. AES Encryption Demo
3. RSA Key Exchange
4. Hybrid Encryption (AES + RSA)
5. JWT Authentication
6. End-to-End Flow Simulation
7. Attack Simulations
8. Performance Benchmarking
9. Conclusion

1) Symmetric-key design ⇒ low computational overhead

AES (Advanced Encryption Standard) is a symmetric block cipher: the same secret key is used for encryption and decryption.
Unlike asymmetric cryptography (e.g., RSA cryptosystem), AES avoids expensive operations like modular exponentiation over large integers. Instead, it uses simple byte-level and word-level transformations.

Consequence: far fewer CPU cycles per byte ⇒ high throughput and low latency.

2) Efficient round function on modern CPUs

AES operates on 128-bit blocks with key sizes of 128/192/256 bits. Each round applies:

SubBytes (nonlinear substitution via an S-box)
ShiftRows (byte permutation)
MixColumns (linear transformation over GF(2⁸))
AddRoundKey (XOR with round key)

These are composed of table lookups, XORs, and finite-field arithmetic—all very fast on general-purpose processors.

3) Hardware acceleration (AES-NI)

Modern CPUs implement AES-NI (AES New Instructions), a set of dedicated instructions that execute AES rounds in hardware.

Effects:

Orders-of-magnitude speedup vs software-only implementations
Near-constant-time execution paths ⇒ mitigates certain side-channel risks
Multi-GB/s throughput on commodity hardware

4) Designed for bulk data encryption

AES is a block cipher, so it’s used with modes of operation (e.g., CBC mode, CTR mode, GCM) to securely process arbitrarily large data streams.

CTR/GCM turn AES into a stream-like construction, enabling parallelism and high throughput.
GCM additionally provides AEAD (Authenticated Encryption with Associated Data)—confidentiality + integrity in one pass.

Consequence: ideal for encrypting large files, databases, backups, and network traffic.

5) Parallelization and pipeline efficiency

Several modes (notably CTR and GCM) allow block-level parallelism:

Multiple blocks can be encrypted simultaneously (SIMD/vectorization)
Works well with CPU pipelines and multicore systems

Result: scales efficiently with hardware ⇒ excellent for high-volume workloads.

6) Strong security margin with practical performance

AES is standardized by National Institute of Standards and Technology and has withstood extensive cryptanalysis.

AES-128/192/256 provide strong security levels
No practical attacks on full-round AES
Well-understood security properties when used with correct modes (e.g., GCM)

Balance: high assurance without sacrificing speed.

7) Typical system design (hybrid encryption)

In real systems, AES is paired with asymmetric crypto:

Use RSA cryptosystem or Elliptic-curve Diffie–Hellman to securely establish a session key
Use AES with that session key to encrypt the bulk data

Rationale: asymmetric crypto is slow but solves key distribution; AES is fast and handles the data.

8) Performance characteristics (rule-of-thumb)
Time complexity: O(n) in data size with a small constant factor
Throughput: hundreds of MB/s to multiple GB/s with AES-NI
Memory footprint: minimal; operates on fixed-size state (128-bit blocks)

In [ ]:
from cryptography.fernet import Fernet

# Generate key
aes_key = Fernet.generate_key()
cipher = Fernet(aes_key)

message = b"Sensitive banking data"

# Encrypt
encrypted = cipher.encrypt(message)

# Decrypt
decrypted = cipher.decrypt(encrypted)

print("AES Encrypted:", encrypted)
print("AES Decrypted:", decrypted)

AES Encrypted: b'gAAAAABp6g6EhgqFVMW8GWg6b8p6osxRpd16sxaygfxXX4v2LiYxLEPp-9ilJ8WQhm6ddj8AcusL_Yz6IQ86E25N9DHkgyXwL6BT1hUhL_IhhBH_KecoK88='
AES Decrypted: b'Sensitive banking data'


Why AES alone is not enough (the key distribution problem)

AES is a symmetric cipher: the same secret key must be known to both sender and receiver.

That creates a fundamental issue:

How do you securely share the AES key in the first place?

If we

Send the AES key over the network → it can be intercepted
Store it somewhere shared → it becomes a single point of compromise
Hardcode it → not scalable, not secure

This is called the key distribution problem.

So even though AES is:

Fast
Secure (cryptographically strong)
…it doesn’t solve secure key exchange.

Solution: Hybrid Cryptosystem (AES + RSA)

This is where RSA cryptosystem comes in.

We combine:

AES → encrypts the actual data (fast, efficient)
RSA → securely shares the AES key (solves distribution)

In [ ]:
from cryptography.hazmat.primitives.asymmetric import rsa, padding
from cryptography.hazmat.primitives import serialization, hashes

# Generate keys
private_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
public_key = private_key.public_key()

# Encrypt AES key using RSA
encrypted_key = public_key.encrypt(
    aes_key,
    padding.OAEP(mgf=padding.MGF1(algorithm=hashes.SHA256()),
                 algorithm=hashes.SHA256(),
                 label=None)
)

# Decrypt AES key
decrypted_key = private_key.decrypt(
    encrypted_key,
    padding.OAEP(mgf=padding.MGF1(algorithm=hashes.SHA256()),
                 algorithm=hashes.SHA256(),
                 label=None)
)

print(decrypted_key == aes_key)

True


Hybrid Encryption (REAL-WORLD SYSTEM):
Hybrid encryption is a combination of symmetric and asymmetric cryptography, typically modeled as KEM-DEM. A random session key is generated and used with a symmetric cipher like AES-GCM to efficiently encrypt bulk data. That session key is then securely encapsulated using the recipient’s public key via RSA-OAEP or ECDH. This approach provides both high performance and secure key distribution, and is widely used in protocols like TLS and cloud encryption systems. Used at Indusind Bank Limited.

In [ ]:
# Sender side
message = b"Transfer 10,00,000"

encrypted_message = cipher.encrypt(message)

# Send: encrypted_message + encrypted_key

In [ ]:
# Receiver side
cipher_receiver = Fernet(decrypted_key)
original_message = cipher_receiver.decrypt(encrypted_message)

print(original_message)

b'Transfer 10,00,000'


JWT (JSON Web Token) authentication is a stateless, token-based authentication mechanism where a client proves its identity by presenting a signed token issued by a trusted authority, instead of relying on server-side session storage.

Core idea
After successful login, the server issues a JWT
The client stores it (usually in memory / cookie / local storage)
Every request includes the token (typically in the Authorization: Bearer <token> header)
The server verifies the token signature and grants access

 No server-side session lookup → stateless authentication

In [ ]:
import jwt
import datetime

secret = "supersecret"

payload = {
    "user": "client_123",
    "exp": datetime.datetime.utcnow() + datetime.timedelta(minutes=5)
}

token = jwt.encode(payload, secret, algorithm="HS256")

# Decode
decoded = jwt.decode(token, secret, algorithms=["HS256"])

print(decoded)

{'user': 'client_123', 'exp': 1776947760}


/tmp/ipykernel_7849/1856437201.py:8: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "exp": datetime.datetime.utcnow() + datetime.timedelta(minutes=5)
/usr/local/lib/python3.12/dist-packages/jwt/api_jwt.py:147: InsecureKeyLengthWarning: The HMAC key is 11 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  return self._jws.encode(
/usr/local/lib/python3.12/dist-packages/jwt/api_jwt.py:365: InsecureKeyLengthWarning: The HMAC key is 11 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  decoded = self.decode_complete(


In [ ]:
def send_message(message):
    encrypted_msg = cipher.encrypt(message)
    encrypted_key = public_key.encrypt(aes_key, padding.OAEP(
        mgf=padding.MGF1(hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    ))
    return encrypted_msg, encrypted_key

def receive_message(enc_msg, enc_key):
    key = private_key.decrypt(enc_key, padding.OAEP(
        mgf=padding.MGF1(hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    ))
    cipher_local = Fernet(key)
    return cipher_local.decrypt(enc_msg)

In [ ]:
tampered = encrypted_message[:-10] + b'1234567890'

try:
    cipher.decrypt(tampered)
except Exception as e:
    print("Tampering detected:", e)

Tampering detected: 


In [ ]:
fake_key = Fernet.generate_key()
fake_cipher = Fernet(fake_key)

try:
    fake_cipher.decrypt(encrypted_message)
except:
    print("Decryption failed with wrong key")

Decryption failed with wrong key


In [ ]:
tampered_token = token + "abc"

try:
    jwt.decode(tampered_token, secret, algorithms=["HS256"])
except:
    print("Invalid token detected")

Invalid token detected


In [ ]:
import time

start = time.time()
for _ in range(1000):
    cipher.encrypt(message)
print("AES time:", time.time() - start)

AES time: 0.03785967826843262
